# ResNet18-Encoder U-Net (Montgomery CXR)

Training, metrics, and Phase 2/3 interpretability code live in the **`unet_mech`** package. This notebook is a thin entry point; use the same imports from Colab after installing requirements and adding the repo root to `sys.path`.

- **Train:** `python scripts/train_baseline.py` (see `--help`; use `--smoke` for a 2-epoch sanity run).
- **Bottleneck visualisation:** `python scripts/run_bottleneck_viz.py --ckpt resnet18_unet_best.pth`
- **Channel ablation:** `python scripts/run_ablation.py --ckpt resnet18_unet_best.pth --from-ch 0 --to-ch 8 --out-csv outputs/ablation.csv`

In [ ]:
# Optional: pip install -r requirements.txt
# !pip install -r requirements.txt

In [ ]:
import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from unet_mech.config import DEFAULT_CFG

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("config keys:", list(DEFAULT_CFG.keys()))

In [ ]:
# Example: data loaders (requires Montgomery extracted under data_dir)
import torch
from unet_mech.config import DEFAULT_CFG
from unet_mech.data import build_dataloaders, download_montgomery

cfg = DEFAULT_CFG.copy()
cfg["device"] = "cuda" if torch.cuda.is_available() else "cpu"
root = download_montgomery(cfg["data_dir"])
train_loader, val_loader, test_loader = build_dataloaders(
    root=str(root),
    batch_size=cfg["batch_size"],
    img_size=cfg["img_size"],
    train_frac=cfg["train_frac"],
    val_frac=cfg["val_frac"],
    seed=cfg["seed"],
)
len(train_loader), len(val_loader), len(test_loader)

In [ ]:
# Full training from the project root (terminal):
#   python scripts/train_baseline.py
# Or:
#   !cd {ROOT} && python scripts/train_baseline.py --smoke

In [ ]:
# Phase 2/3: run after data cell above (needs val_loader) and optional checkpoint
import torch
from unet_mech.models import ResNet18UNet
from unet_mech.interpret.bottleneck_viz import save_bottleneck_channel_grid, save_bottleneck_with_overlay
from unet_mech.interpret.ablation import ablation_sweep_bottleneck

CKPT = "resnet18_unet_best.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ResNet18UNet(pretrained=True).to(device)
# model.load_state_dict(torch.load(CKPT, map_location=device)["state_dict"])

images, _ = next(iter(val_loader))
save_bottleneck_channel_grid(model, images[0], device, save_path="outputs/bottleneck_grid.png", max_channels=16)
save_bottleneck_with_overlay(model, images[0], device, channel=0, save_path="outputs/bottleneck_overlay_ch0.png")
# ablation_sweep_bottleneck(model, val_loader, device, list(range(0, 4)), csv_path="outputs/ablation.csv")